In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('goldman_sachs.csv')

In [4]:

txn_count = df.groupby('AccountID').size().reset_index(name='TxnCount')

def activity_level(x):
    if x > 20:
        return 'High'
    elif x >= 10:
        return 'Medium'
    else:
        return 'Low'


txn_count['ActivityLevel'] = txn_count['TxnCount'].apply(activity_level)

print(txn_count)


    AccountID  TxnCount ActivityLevel
0    ACC10117         4           Low
1    ACC10996         5           Low
2    ACC11062         2           Low
3    ACC11188         5           Low
4    ACC11285         3           Low
..        ...       ...           ...
189  ACC97225         3           Low
190  ACC97411         2           Low
191  ACC99117         3           Low
192  ACC99409         4           Low
193  ACC99549         4           Low

[194 rows x 3 columns]


In [5]:
txn_count['ActivityLevel'].value_counts()

ActivityLevel
Low       191
Medium      3
Name: count, dtype: int64

In [9]:
account_summary = df.groupby('AccountID').agg(
    TxnCount=('TransactionAmount', 'count'),
    AvgBalance=('AccountBalance', 'mean'),
    Credits=('TransactionAmount', lambda x: x[x > 0].sum()),
    Debits=('TransactionAmount', lambda x: abs(x[x < 0].sum()))
)

account_summary['NetInflow'] = (
    account_summary['Credits'] - account_summary['Debits']
)

account_summary = account_summary.reset_index()


In [17]:
threshold = account_summary['NetInflow'].quantile(0.75)

high_net_inflow = account_summary[
    account_summary['NetInflow'] > threshold
]

print(high_net_inflow)

    AccountID  TxnCount    AvgBalance        Credits        Debits  \
7    ACC12334         6  78082.517883  310227.289690      0.000000   
8    ACC13357         6  69179.806513  432527.808260      0.000000   
13   ACC16241        10  73521.710375  539612.142850      0.000000   
29   ACC23736         7  60801.533102  309797.642870      0.000000   
42   ACC28292        10  51228.003570  368846.147320   7628.914828   
48   ACC29356         9  91799.550571  435573.496188      0.000000   
49   ACC29396         8  80302.140096  482720.015550      0.000000   
55   ACC31539         6  45185.938342  402764.370600      0.000000   
56   ACC31902         5  75207.818822  331196.607580      0.000000   
57   ACC32212         6  66264.686983  384045.682620      0.000000   
58   ACC32627         6  82865.126973  337291.337800      0.000000   
60   ACC33287         8  59331.981186  591591.095890      0.000000   
68   ACC37688         7  63580.556277  421852.929240      0.000000   
73   ACC39529       

In [16]:
avg_txn = account_summary['TxnCount'].mean()
avg_balance = account_summary['AvgBalance'].mean()

high_freq_low_balance = account_summary[
    (account_summary['TxnCount'] > avg_txn) &
    (account_summary['AvgBalance'] < avg_balance)
]

print(high_freq_low_balance)

    AccountID  TxnCount    AvgBalance        Credits        Debits  \
1    ACC10996         5  43568.008084  250739.550950      0.000000   
3    ACC11188         5  69652.151044  257576.603590      0.000000   
8    ACC13357         6  69179.806513  432527.808260      0.000000   
18   ACC18177         5  63505.219474  249580.485430      0.000000   
29   ACC23736         7  60801.533102  309797.642870      0.000000   
31   ACC24070         5  55694.967801  258323.636250      0.000000   
40   ACC26973         5  58738.210687  289026.317264      0.000000   
42   ACC28292        10  51228.003570  368846.147320   7628.914828   
53   ACC30787         5  60525.872356  280526.017831      0.000000   
55   ACC31539         6  45185.938342  402764.370600      0.000000   
57   ACC32212         6  66264.686983  384045.682620      0.000000   
60   ACC33287         8  59331.981186  591591.095890      0.000000   
68   ACC37688         7  63580.556277  421852.929240      0.000000   
83   ACC45101       

In [13]:
negative_near_zero = account_summary[
    account_summary['AvgBalance'] <= 1000
]

print(negative_near_zero)

   AccountID  TxnCount   AvgBalance      Credits  Debits    NetInflow
20  ACC19178         1 -1541.176812  64100.78213     0.0  64100.78213
